# Возвращение нелинейного маятника

В зелёной работе начальную скорость подбирали для линейного уравнения. Теперь решим задачу
$$\ddot\theta+\sin\theta=0,\qquad \theta(0)=\theta(4)=0.$$

Начальная скорость $s=\dot\theta(0)$ неизвестна. Найдём решения при $s\in[-1.99,1.99]$, сравним их движение и проверим, откуда к ним сходится метод Ньютона. Время безразмерное. Конечный угол должен быть равен именно нулю: полный оборот не считается выполнением условия.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from IPython.display import HTML, display
import re

plt.rcParams.update({
    "figure.dpi": 85,
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "animation.embed_limit": 60,
})

T = 4.0
S_LIMIT = 1.99
ROOT_TOL = 1e-12
h_work = 0.005
colors = ["#2563eb", "#6b7280", "#df7620"]


## 1. Невязка метода стрельбы

Для каждой пробной скорости решаем задачу Коши
$$\dot\theta=v,\qquad \dot v=-\sin\theta,\qquad \theta(0)=0,\quad v(0)=s.$$
Невязка — угол в конце расчёта: $R(s)=\theta(4;s)$. Искомые скорости соответствуют её нулям.

Используем RK4 из зелёного уровня, заменив ускорение $-\theta$ на $-\sin\theta$. При построении графика функция считает много начальных скоростей одновременно. Сам угол при интегрировании не приводится по модулю $2\pi$.


In [ ]:
def rk4_step(theta, v, h):
    k1t, k1v = v, -np.sin(theta)
    k2t = v + h * k1v / 2
    k2v = -np.sin(theta + h * k1t / 2)
    k3t = v + h * k2v / 2
    k3v = -np.sin(theta + h * k2t / 2)
    k4t = v + h * k3v
    k4v = -np.sin(theta + h * k3t)
    theta_new = theta + h * (k1t + 2*k2t + 2*k3t + k4t) / 6
    v_new = v + h * (k1v + 2*k2v + 2*k3v + k4v) / 6
    return theta_new, v_new


def solve(s, h, save=False):
    s = np.asarray(s, dtype=float)
    n = int(np.ceil(T / h))
    dt = T / n
    theta = np.zeros_like(s)
    v = s.copy()
    if save:
        angles = np.empty((n + 1,) + s.shape)
        velocities = np.empty_like(angles)
        angles[0], velocities[0] = theta, v
    for i in range(n):
        theta, v = rk4_step(theta, v, dt)
        if save:
            angles[i + 1], velocities[i + 1] = theta, v
    if save:
        return np.linspace(0, T, n + 1), angles, velocities
    return theta


def residual(s, h=h_work):
    return solve(s, h)


def bisection(a, b, h):
    ra, rb = float(residual(a, h)), float(residual(b, h))
    if abs(ra) <= ROOT_TOL:
        return a
    if abs(rb) <= ROOT_TOL:
        return b
    if ra * rb >= 0:
        raise ValueError("На концах интервала нужна смена знака невязки.")
    for _ in range(80):
        c = (a + b) / 2
        rc = float(residual(c, h))
        if abs(rc) <= ROOT_TOL:
            return c
        if ra * rc < 0:
            b = c
        else:
            a, ra = c, rc
    raise RuntimeError("Бисекция не достигла допуска.")


def find_roots(h):
    grid = np.linspace(-S_LIMIT, S_LIMIT, 801)
    grid[400] = 0.0
    values = residual(grid, h)
    roots = list(grid[np.abs(values) <= ROOT_TOL])
    for i in np.flatnonzero(values[:-1] * values[1:] < 0):
        roots.append(bisection(grid[i], grid[i + 1], h))
    return np.array(sorted(roots))


## 2. Корни и соответствующие движения

Сначала просматриваем невязку на сетке скоростей. Интервалы со сменой знака уточняем бисекцией; точный ноль $s=0$ учитываем отдельно. Это позволяет найти корни независимо от того, насколько удачно выбрано начальное приближение для Ньютона.


In [ ]:
roots = find_roots(h_work)
assert len(roots) == 3
s_grid = np.linspace(-S_LIMIT, S_LIMIT, 1001)
R_grid = residual(s_grid)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(s_grid, R_grid, color="#39424e", label="R(s) = θ(4; s)")
ax.axhline(0, color="black", lw=0.8)
for s, color in zip(roots, colors):
    ax.plot(s, residual(s), "o", color=color, ms=7, zorder=3,
            label=f"s = {s:.6f}")
ax.set(xlabel="Начальная скорость s", ylabel="Невязка R(s), рад",
       title="Нули невязки", xlim=(-S_LIMIT, S_LIMIT))
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

t, theta, velocity = solve(roots, h_work, save=True)
print("       s          θ(4)          v(4)       max |θ|, рад")
for j, s in enumerate(roots):
    print(f"{s: .10f}   {theta[-1, j]: .2e}   {velocity[-1, j]: .8f}     {np.max(np.abs(theta[:, j])):.6f}")


In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for j, (s, color) in enumerate(zip(roots, colors)):
    style = "--" if j == 1 else "-"
    ax[0].plot(t, theta[:, j], style, color=color, label=f"s = {s:.6f}")
    ax[1].plot(t, velocity[:, j], style, color=color)
ax[0].scatter([0, T], [0, 0], marker="*", color="black", s=80, zorder=5,
              label="Заданные углы")
ax[0].set(ylabel="Угол θ, рад", title="Три движения с одинаковыми крайними положениями")
ax[1].set(xlabel="Время t", ylabel="Скорость v")
ax[0].legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=9)
fig.tight_layout()
plt.show()


Получились покой и два движения с противоположными начальными скоростями $s\approx\pm1.604813$. Ненулевые решения симметричны: замена $s$ на $-s$ меняет знаки угла и скорости.

Маятник с положительной начальной скоростью отклоняется в одну сторону, останавливается и возвращается к нулевому углу в момент $t=4$. При возвращении скорость отрицательна. Для отрицательной начальной скорости всё происходит зеркально.

При $|s|<2$ энергия $E=s^2/2<2$, поэтому маятник колеблется без полных оборотов. Возврат к нулю занимает половину периода. Здесь период ненулевых решений равен 8. Период растёт с амплитудой от $2\pi$ до бесконечности, поэтому для времени возврата 4 есть одна положительная скорость и одна отрицательная. Успеть вернуться дважды за это время нельзя: даже два самых коротких полупериода занимают $2\pi>4$.


## 3. Зависимость Ньютона от начального приближения

Используем итерацию
$$s_{k+1}=s_k-\frac{R(s_k)}{R'(s_k)}.$$
Производную, как в зелёной работе, вычисляем центральной разностью. Останавливаемся при $|R|\leq10^{-12}$, слишком малой производной, выходе за $[-1.99,1.99]$ или после 30 итераций. Выход за интервал — выбранное здесь правило остановки метода.

Запустим один алгоритм из 1001 равномерно расположенного начального приближения. На карте цвет обозначает найденный корень или причину остановки. Повторим расчёт с вдвое меньшим шагом интегрирования.


In [ ]:
status_names = {
    1: "Сошёлся",
    2: "Малая производная",
    3: "Выход за интервал",
    4: "Лимит итераций",
    5: "Нечисловое значение",
}


def newton(starts, h, max_iterations=30):
    s = np.atleast_1d(starts).astype(float).copy()
    status = np.zeros(s.size, dtype=int)
    iterations = np.zeros(s.size, dtype=int)
    status[~np.isfinite(s)] = 5
    status[np.isfinite(s) & (np.abs(s) > S_LIMIT)] = 3
    for _ in range(max_iterations + 1):
        active = np.flatnonzero(status == 0)
        if active.size == 0:
            break
        r = residual(s[active], h)
        converged = np.abs(r) <= ROOT_TOL
        status[active[converged]] = 1
        active, r = active[~converged], r[~converged]
        if active.size == 0:
            continue
        limited = iterations[active] >= max_iterations
        status[active[limited]] = 4
        active, r = active[~limited], r[~limited]
        if active.size == 0:
            continue
        delta = np.cbrt(np.finfo(float).eps) * np.maximum(1, np.abs(s[active]))
        derivative = (residual(s[active] + delta, h)
                      - residual(s[active] - delta, h)) / (2 * delta)
        small = np.abs(derivative) < 1e-10
        status[active[small]] = 2
        active, r, derivative = active[~small], r[~small], derivative[~small]
        candidate = s[active] - r / derivative
        finite = np.isfinite(candidate)
        status[active[~finite]] = 5
        outside = finite & (np.abs(candidate) > S_LIMIT)
        status[active[outside]] = 3
        accepted = finite & ~outside
        s[active[accepted]] = candidate[accepted]
        iterations[active[accepted]] += 1
    return {"s": s, "status": status, "iterations": iterations}


category_names = ["Корень s < 0", "Покой, s = 0", "Корень s > 0",
                  "Выход за интервал", "Малая производная",
                  "Лимит итераций", "Нечисловое значение"]
category_colors = colors + ["#c33d4a", "#8848b8", "#171717", "#d6bd35"]


def classify(result):
    category = np.full(result["s"].size, -1, dtype=int)
    ok = result["status"] == 1
    nearest = np.argmin(np.abs(result["s"][ok, None] - roots), axis=1)
    assert np.all(np.abs(result["s"][ok] - roots[nearest]) < 1e-7)
    category[ok] = nearest
    for status, value in [(3, 3), (2, 4), (4, 5), (5, 6)]:
        category[result["status"] == status] = value
    assert np.all(category >= 0)
    return category


In [ ]:
starts = np.linspace(-S_LIMIT, S_LIMIT, 1001)
starts[500] = 0.0
map_steps = [0.02, 0.01]
maps = {h: newton(starts, h) for h in map_steps}
categories = {h: classify(maps[h]) for h in map_steps}
cmap = ListedColormap(category_colors)
norm = BoundaryNorm(np.arange(-0.5, 7.5), cmap.N)
edges = np.r_[starts[0] - (starts[1] - starts[0])/2,
              (starts[:-1] + starts[1:])/2,
              starts[-1] + (starts[-1] - starts[-2])/2]

fig, axes = plt.subplots(3, 1, figsize=(10, 5.4), sharex=True,
                         gridspec_kw={"height_ratios": [0.45, 0.45, 1.3]})
fig.subplots_adjust(left=0.13, right=0.97, top=0.89, bottom=0.25, hspace=0.25)
fig.suptitle("Результат Ньютона при разных начальных приближениях")
for a, h in zip(axes[:2], map_steps):
    a.pcolormesh(edges, [0, 1], categories[h][None, :], cmap=cmap, norm=norm)
    a.set_yticks([])
    a.set_ylabel(f"h={h:g}", rotation=0, ha="right", va="center", labelpad=10)
    a.grid(False)
result = maps[0.01]
axes[2].scatter(starts, result["iterations"], c=categories[0.01], cmap=cmap, norm=norm,
                s=7, linewidths=0)
axes[2].set(xlim=(-S_LIMIT, S_LIMIT), xlabel="Начальное приближение s₀",
            ylabel="Принятые шаги", ylim=(-0.5, result["iterations"].max() + 1))
observed = np.unique(np.concatenate(list(categories.values())))
fig.legend(handles=[Patch(facecolor=category_colors[k], label=category_names[k]) for k in observed],
           loc="lower center", bbox_to_anchor=(0.5, 0.01), ncol=2, fontsize=9)
plt.show()

print("Результат на сетке из 1001 начального приближения, h=0.01:")
for k in observed:
    print(f"{category_names[k]}: {np.count_nonzero(categories[0.01] == k)}")
print("Изменившихся цветов после уменьшения h вдвое:",
      np.count_nonzero(categories[0.02] != categories[0.01]))

examples = np.array([0.0, 0.5, 1.0, 1.2, 1.5, 1.9])
example_result = newton(examples, 0.01)
print("\n   s₀       Найденный корень / причина остановки      Шагов")
for j, start in enumerate(examples):
    status = example_result["status"][j]
    answer = (f's = {example_result["s"][j]: .9f}' if status == 1 else status_names[status])
    print(f"{start:6.2f}      {answer:35s}  {example_result['iterations'][j]:3d}")


Из $s_0=0.5$ метод приходит к покою, а из $s_0=1.5$ — к положительному ненулевому решению. Для $s_0=1$ и $1.2$ уже первый шаг выходит за исследуемый интервал. Рядом с экстремумом невязки производная мала, поэтому поправка Ньютона может оказаться большой.

На выбранной сетке уменьшение шага с $0.02$ до $0.01$ не изменило цвета карты. Это проверка именно этих 1001 стартовых значений; около границ цветных участков более густая сетка может дать дополнительные детали.


## 4. Проверка уменьшением шага

Для каждого шага заново найдём все три скорости. Затем сравним движения с опорным расчётом при $h=0.00125$. В таблице покажем положительный корень, невязку на его собственной сетке и невязку при повторном интегрировании с шагом $h/2$ без изменения найденной скорости.

Последний столбец — максимальное отличие углов от опорного решения в общих узлах времени. Каждый расчёт использует свою заново подобранную скорость. Отрицательная ветвь даёт те же абсолютные ошибки по симметрии.


In [ ]:
steps = [0.16, 0.08, 0.04, 0.02, 0.01, 0.005]
h_ref = 0.00125
reference_roots = find_roots(h_ref)
t_ref, theta_ref, _ = solve(reference_roots, h_ref, save=True)
angle_errors = []
refined_residuals = []

print("    h          s > 0         |R_h|       |R_(h/2)|     max |θ_h − θ_ref|")
for h in steps:
    rr = find_roots(h)
    assert len(rr) == 3 and np.allclose(rr, -rr[::-1], atol=1e-10)
    tt, yy, _ = solve(rr, h, save=True)
    stride = int(round(h / h_ref))
    assert np.allclose(tt, t_ref[::stride], atol=1e-12, rtol=0)
    error = np.max(np.abs(yy - theta_ref[::stride]))
    own_residual = np.max(np.abs(yy[-1]))
    refined = np.max(np.abs(residual(rr, h / 2)))
    angle_errors.append(error)
    refined_residuals.append(refined)
    print(f"{h:7.3f}   {rr[-1]:.11f}   {own_residual:9.2e}   {refined:11.2e}       {error:11.2e}")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].loglog(steps, angle_errors, "o-", color=colors[0])
ax[0].loglog(steps, angle_errors[0] * (np.array(steps) / steps[0])**4,
             "--", color="0.5", label="Масштаб h⁴")
ax[1].loglog(steps, refined_residuals, "o-", color=colors[2])
ax[0].set(ylabel="max |θ_h − θ_ref|, рад", title="Отличие движений")
ax[1].set(ylabel=r"$|\theta_{h/2}(4; s_h)|$, рад", title="Проверка скоростей на мелкой сетке")
for a in ax:
    a.invert_xaxis()
    a.set_xlabel("Шаг h (уменьшается →)")
    a.set_xticks(steps, [f"{h:g}" for h in steps])
    a.minorticks_off()
ax[0].legend(fontsize=9)
fig.tight_layout()
plt.show()


Малая невязка на собственной сетке получается при любом из рассмотренных шагов: бисекция подбирает скорость именно для этого численного решения. Повторное интегрирование с меньшим шагом показывает оставшуюся ошибку. При уточнении сетки сходятся и начальные скорости, и сами движения.

Для анимации возьмём $h=0.005$. Максимальное отличие угла от опорного расчёта здесь около $8\cdot10^{-12}$ рад, тогда как при $h=0.16$ оно было около $8\cdot10^{-6}$ рад.



## 5. Анимация трёх решений

Сверху показаны три маятника, снизу — их углы на одной временной шкале. Цвета совпадают с предыдущими графиками. Вертикальная линия и точки отмечают текущий момент. В конце все грузы возвращаются в нижнее положение, но только средний маятник остаётся неподвижным.


In [ ]:
def show_animation(animation):
    html = animation.to_jshtml(fps=20, default_mode="once")
    html = re.sub(r'<link\b[^>]*font-awesome[^>]*>', '', html)
    icons = {
        "fa fa-minus": "−", "fa fa-plus": "+",
        "fa fa-fast-backward": "⏮", "fa fa-step-backward": "⏪",
        "fa fa-play fa-flip-horizontal": "◀", "fa fa-play": "▶",
        "fa fa-pause": "⏸", "fa fa-step-forward": "⏩",
        "fa fa-fast-forward": "⏭",
    }
    for css, symbol in icons.items():
        html = html.replace(f'<i class="{css}"></i>', f'<span>{symbol}</span>')
    display(HTML(html))


frame_ids = np.arange(0, len(t), int(round(0.05 / h_work)))
fig = plt.figure(figsize=(10, 6.2))
grid = fig.add_gridspec(2, 3, height_ratios=[1.15, 1])
pendulum_axes = [fig.add_subplot(grid[0, j]) for j in range(3)]
angle_ax = fig.add_subplot(grid[1, :])
rods, bobs = [], []
for j, a in enumerate(pendulum_axes):
    a.plot(0, 0, "ko", ms=4, zorder=4)
    a.plot([0, 0], [0, -1], ":", color="0.75", lw=1)
    rod, = a.plot([], [], color=colors[j], lw=2.5)
    bob, = a.plot([], [], "o", color=colors[j], ms=12)
    rods.append(rod)
    bobs.append(bob)
    a.set(xlim=(-1.15, 1.15), ylim=(-1.15, 0.5),
          xlabel="x / L", ylabel="y / L", title=f"s = {roots[j]:.6f}")
    a.set_aspect("equal")
points = []
for j in range(3):
    angle_ax.plot(t, theta[:, j], "--" if j == 1 else "-",
                  color=colors[j], label=f"s = {roots[j]:.6f}")
    point, = angle_ax.plot([], [], "o", color=colors[j], ms=6)
    points.append(point)
cursor = angle_ax.axvline(0, color="0.35", ls="--", lw=1)
angle_ax.set(xlim=(0, T), xlabel="Время t", ylabel="Угол θ, рад")
angle_ax.legend(loc="upper right", fontsize=8)
clock = fig.suptitle("t = 0.00", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.96])


def update(k):
    i = frame_ids[k]
    for j in range(3):
        x, y = np.sin(theta[i, j]), -np.cos(theta[i, j])
        rods[j].set_data([0, x], [0, y])
        bobs[j].set_data([x], [y])
        points[j].set_data([t[i]], [theta[i, j]])
    cursor.set_xdata([t[i], t[i]])
    clock.set_text(f"t = {t[i]:.2f}")
    return *rods, *bobs, *points, cursor, clock


animation = FuncAnimation(fig, update, frames=len(frame_ids), interval=50)
plt.close(fig)
show_animation(animation)


## Вывод

Одинаковые углы в начале и конце не задают скорости маятника и весь путь между этими моментами. В этой задаче можно оставаться в покое либо отклониться в одну из двух сторон и вернуться через четыре единицы времени.

Получены три начальные скорости: $s=0$ и $s\approx\pm1.604813$. Найденные движения сохраняются при уменьшении шага. Метод Ньютона выбирает решение в зависимости от старта, а для части начальных приближений останавливается из-за выхода за исследуемый интервал.
